# Chess Model Training on Google Colab

This notebook is configured to train the chess model using the Google Colab GPU runtime.
It uses the configuration from `config/gpu_training.yaml` and targets `data/chess_dataset_new.db`.

## 1. Setup Environment
Mount Google Drive and install dependencies.

In [1]:
import os
import sys
from pathlib import Path

# 1. Mount Google Drive (if in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")
    
    # Attempt to find project path in Drive (MyDrive or Computers)
    possible_paths = [
        '/content/drive/Othercomputers/My MacBook Pro/project',
        '/content/drive/Othercomputers/my macbook pro/project',
        '/content/drive/MyDrive/FYP/project',
    ]
    
    project_path = None
    for path in possible_paths:
        if os.path.exists(path):
            project_path = path
            break
            
    if project_path:
        os.chdir(project_path)
        print(f"Found project at: {project_path}")
        print(f"Changed working directory to: {os.getcwd()}")
    else:
        print("WARNING: Could not find project directory. Checked:", possible_paths)
        print("Current directory:", os.getcwd())
        print("Please list contents of /content/drive to find your path.")
        
except ImportError:
    print("Not running in Google Colab environment. Skipping Drive mount.")
    print(f"Current working directory: {os.getcwd()}")

# 2. Add current directory to sys.path
cwd = str(Path.cwd())
if cwd not in sys.path:
    sys.path.append(cwd)
    print(f"Added {cwd} to sys.path")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.
Found project at: /content/drive/Othercomputers/My MacBook Pro/project
Changed working directory to: /content/drive/Othercomputers/My MacBook Pro/project
Added /content/drive/Othercomputers/My MacBook Pro/project to sys.path


In [2]:
# 3. Install Dependencies
# We use 'pip install' with quiet flag, remove -q to see output
if os.path.exists('requirements.txt'):
    print("Installing dependencies from requirements.txt...")
    !pip install -r requirements.txt
else:
    print("requirements.txt not found. Installing default packages...")
    !pip install python-chess torch torch-geometric pyyaml numpy tqdm

Installing dependencies from requirements.txt...


## 2. Load Configuration & Data

In [3]:
import torch
import yaml

try:
    from src.config import load_config
    from src.device import get_device_from_config
    from src.models.factory import create_model, get_encoder_for_model
    from src.data.dataset import create_dataloader
    from src.training import Trainer
    print("Successfully imported src modules.")
except ImportError as e:
    print(f"ERROR: {e}")
    print("Ensure you are in the correct directory (the project root) and 'src' folder exists.")
    print(f"Contents of current dir ({os.getcwd()}): {os.listdir('.')}")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

Successfully imported src modules.
PyTorch version: 2.9.0+cu126
CUDA available: True
Device: Tesla T4


In [4]:
# Load Configuration
config_path = "config/gpu_training.yaml"
if os.path.exists(config_path):
    print(f"Loading config from {config_path}")
    config = load_config(config_path)
else:
    print(f"Config file {config_path} not found. Using defaults.")
    raise FileNotFoundError(f"Config not found at {config_path}")

# Verify Database
db_path = "data/chess_dataset_new.db"
if not os.path.exists(db_path):
    print(f"WARNING: Database not found at {db_path}.")
    # Attempt to use absolute path if in Drive
    abs_db_path = os.path.join(os.getcwd(), db_path)
    if os.path.exists(abs_db_path):
         print(f"Found at absolute path: {abs_db_path}")
         db_path = abs_db_path
    else:
         print("Please ensure the database file is uploaded to data/chess_dataset_new.db")

print(f"Using database: {db_path}")
config.paths.database = db_path

# Colab specific adjustments
config.hardware.num_workers = 2 # Prevent shared memory issues in Colab

Loading config from config/gpu_training.yaml
Using database: data/chess_dataset_new.db


## 3. Initialize Model

In [5]:
device_config = {"hardware": {"device": "cuda" if torch.cuda.is_available() else "cpu"}}
device = get_device_from_config(device_config)

print(f"Creating model: {config.model.backbone} ({config.model.head} head)")
model = create_model(config.model)
print(f"Model parameters: {model.count_parameters():,}")

Creating model: resnet (dual head)
Model parameters: 13,187,777


## 4. Prepare DataLoader

In [ ]:
print("Initializing data loader...")
encoder_factory = get_encoder_for_model(config.model.backbone)
encoder = encoder_factory()

train_loader = create_dataloader(
    db_path=db_path,
    encoder=encoder,
    batch_size=config.training.batch_size,
    shuffle=True,
    num_workers=config.hardware.num_workers,
    use_soft_labels=True,
    include_value=(config.model.head in ["value", "dual"]),
)
print(f"DataLoader created. Dataset size: {len(train_loader.dataset)} positions")

Initializing data loader...


## 5. Train

In [ ]:
checkpoint_dir = "checkpoints/colab_run"
os.makedirs(checkpoint_dir, exist_ok=True)

trainer = Trainer(
    model=model,
    device=device,
    head_type=config.model.head,
    learning_rate=config.training.learning_rate,
    weight_decay=config.training.weight_decay,
    policy_weight=config.training.policy_loss_weight,
    value_weight=config.training.value_loss_weight,
    use_soft_labels=True,
    checkpoint_dir=checkpoint_dir,
)

print("Starting training...")
history = trainer.train(
    train_loader=train_loader,
    val_loader=None,
    epochs=config.training.epochs,
    scheduler_type=config.training.lr_scheduler.type,
    save_best=True,
    save_every=5,
)
print("Training complete!")